# Módulo 3 · Pipelines y workflows con LangChain

**Duración sugerida:** 4 horas · **Entrega:** pipeline ejecutable con instalación,
ejemplos y pruebas básicas.

## Objetivos del laboratorio

- Distinguir **una llamada a un LLM** de un **workflow**.
- Usar componentes de **LangChain**: plantillas de prompts, modelos de chat,
  parsers y salidas estructuradas.
- **Encadenar pasos** con LCEL (LangChain Expression Language).
- Añadir **validación y manejo de errores**.
- Introducir **rutas condicionales** con LangGraph.

## Conexión con el proyecto ARAS

El pipeline de ARAS recibe una pregunta, identifica el tipo de tarea, procesa el
contenido y produce una respuesta estructurada. Aquí construimos ese esqueleto.

## 1. Dependencias

In [ ]:
%pip install -q langchain-core langchain-ollama langgraph pydantic

## 2. Una llamada al LLM vs un workflow

- **Llamada:** prompt → modelo → respuesta (una sola invocación).
- **Workflow:** varios pasos encadenados, con validación, decisiones y reintentos.

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2", temperature=0)

resp = llm.invoke("Define 'workflow' en una frase.")
print(type(resp).__name__, "->", resp.content)

## 3. Plantillas de prompts

`ChatPromptTemplate` separa la estructura del prompt de los datos concretos.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

plantilla = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente académico. Responde en una frase."),
    ("user", "Resume el siguiente texto: {texto}"),
])

mensajes = plantilla.invoke({"texto": "LangChain facilita encadenar modelos, prompts y herramientas."})
print(mensajes)

## 4. Encadenamiento con LCEL

El operador `|` conecta componentes: la salida de uno es la entrada del siguiente.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

cadena = plantilla | llm | StrOutputParser()
print(cadena.invoke({"texto": "Los agentes eligen herramientas y ejecutan acciones para lograr un objetivo."}))

## 5. Salidas estructuradas con un parser Pydantic

Definimos un esquema y pedimos al modelo que devuelva JSON conforme a él.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

class Extraccion(BaseModel):
    titulo: str = Field(description="Título del documento")
    autores: list[str] = Field(description="Lista de autores")
    anio: int = Field(description="Año de publicación")
    hallazgo: str = Field(description="Hallazgo principal en una frase")

parser = JsonOutputParser(pydantic_object=Extraccion)

plantilla_extraccion = ChatPromptTemplate.from_messages([
    ("system", "Extrae la información del texto. Responde SOLO en JSON válido.\n{formato}"),
    ("user", "{texto}"),
]).partial(formato=parser.get_format_instructions())

cadena_extraccion = plantilla_extraccion | llm | parser

texto = (
    "El artículo 'RAG para revisión sistemática', de Carla Díaz y Omar Vidal (2023), "
    "muestra que recuperar fragmentos antes de generar reduce las citas erróneas."
)
resultado = cadena_extraccion.invoke({"texto": texto})
print(resultado)
print(type(resultado).__name__)

## 6. Validación y manejo de errores

Añadimos un reintento con validación: si el JSON no cumple el esquema, repetimos.

In [ ]:
import json

def extraer_seguro(texto, intentos=3):
    for i in range(intentos):
        try:
            out = cadena_extraccion.invoke({"texto": texto})
            if not out.get("titulo") or not out.get("autores"):
                raise ValueError("esquema incompleto")
            return out
        except Exception as e:
            print(f"Intento {i+1} falló: {e}")
    return None

print(extraer_seguro(texto))

## 7. Rutas condicionales con LangGraph (introducción)

LangGraph modela el workflow como un **grafo de estados** con nodos (pasos) y
aristas (transiciones). Aquí, un router decide qué paso ejecutar según el tipo de tarea.

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

class Estado(TypedDict):
    texto: str
    tipo: str
    salida: str

def clasificar(estado: Estado) -> Estado:
    plantilla = ChatPromptTemplate.from_messages([
        ("system", "Clasifica el texto en una sola palabra: PREGUNTA, AFIRMACION u ORDEN."),
        ("user", "{texto}"),
    ])
    cadena = plantilla | llm | StrOutputParser()
    tipo = cadena.invoke({"texto": estado["texto"]}).strip().upper()
    return {"tipo": tipo}

def enrutar(estado: Estado) -> Literal["responder_pregunta", "resumir"]:
    return "responder_pregunta" if estado["tipo"] == "PREGUNTA" else "resumir"

def responder_pregunta(estado: Estado) -> Estado:
    return {"salida": "Respuesta a la pregunta: " + estado["texto"][:60]}

def resumir(estado: Estado) -> Estado:
    return {"salida": "Resumen: " + estado["texto"][:60]}

grafo = StateGraph(Estado)
grafo.add_node("clasificar", clasificar)
grafo.add_node("responder_pregunta", responder_pregunta)
grafo.add_node("resumir", resumir)
grafo.add_edge(START, "clasificar")
grafo.add_conditional_edges("clasificar", enrutar, {"responder_pregunta": "responder_pregunta", "resumir": "resumir"})
grafo.add_edge("responder_pregunta", END)
grafo.add_edge("resumir", END)

app = grafo.compile()
print(app.invoke({"texto": "¿Cómo funciona la atención en los transformers?"}))

## Resumen del laboratorio

- Diferenciamos **llamada al LLM** de **workflow**.
- Usamos **prompt templates, ChatOllama, parsers** y **salidas Pydantic**.
- Encadenamos pasos con **LCEL** y añadimos **reintentos con validación**.
- Introdujimos **LangGraph** con un router condicional.

**Entregable del Módulo 3:** pipeline ejecutable con instalación, ejemplos y pruebas.